# 13.01 · Documents and vision: PDFs, scans and receipts

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lesson 01.02 (images in messages), Lesson 02.03 (structured output) and Module 06 (document loading). Read the module README first. |
| 🔑 Keys | optional: `GROQ_API_KEY` (vision), `GOOGLE_API_KEY` (Gemini free tier, native PDF) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** LangChain standard content blocks for images, PDFs, audio and video, and send the same message to Groq and Gemini.
2. **Check** a multimodal request against provider limits (images per request, size, pages, token cost) *before* it fails with a 400.
3. **Route** each PDF page: read the text layer when there is one, use a vision model only for scans and pictures.
4. **Extract** typed fields (an invoice, a receipt) from a scan with structured output, on Groq vision and on Gemini's native PDF input.
5. **Score** extractions field by field against ground truth, and compare methods on accuracy, tokens and latency.
6. **Close the loop** in a real workflow: match a photographed receipt to the expense ledger and send mismatches to a human.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [LangChain messages: multimodal](https://docs.langchain.com/oss/python/langchain/messages#multimodal) | Standard `image`, `file`, `audio`, `video` blocks (base64, URL, file ID) |
| [Groq vision](https://console.groq.com/docs/vision) | `qwen/qwen3.8-27b` (preview): 3 images per request, 2,048 tokens per image, 20 MB, JSON mode and tools |
| [Gemini document understanding](https://ai.google.dev/gemini-api/docs/document-processing) | Native PDF input: 50 MB or 1,000 pages, 258 tokens per page |
| [ChatGoogleGenerativeAI](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai) | Multimodal input through LangChain (`langchain-google-genai` 4.x) |
| [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing) | Which models are free, and that free-tier content may be used to improve Google's products |

> **Why this matters at work.** Accounts payable, expense reports, insurance claims and onboarding all start with a PDF or a photo. The expensive mistake is not a wrong model, it is sending *every* page to a vision model: most business PDFs already contain their text, which is free and exact to read. This lesson builds the intake step that Project 13 reuses.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain-google-genai>=4.4.1,<5" "pypdf>=6.20.1,<7" "matplotlib>=3.11.2,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-groq", "langchain-google-genai", "pypdf",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm13-documents-and-vision'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

RESULTS = []   # one row per extraction method, filled in Parts 3–6 and compared in Part 7

In [ ]:
# 🧪 Offline: the Module 13 samples (synthetic documents, not real companies).
import json

SAMPLES = REPO_ROOT / "data" / "samples" / "multimodal"
TRUTH = json.loads((SAMPLES / "ground-truth.json").read_text(encoding="utf-8"))
OUT = REPO_ROOT / "data" / "outputs" / "m13"
OUT.mkdir(parents=True, exist_ok=True)
for path in sorted(SAMPLES.iterdir()):
    print(f"{path.name:<22} {path.stat().st_size / 1024:6.0f} KB")

**You should now see** four files: `invoice-digital.pdf` (about 34 KB), `invoice-scanned.pdf` (about 97 KB), `receipt-photo.jpg` (about 32 KB) and `ground-truth.json`.

The invoice is from a fictional data-center vendor to Orbita Cloud (the company from Project 09). The **digital** file is how accounting software exports it; the **scanned** one is the same page printed, scanned slightly crooked and saved as an image-only PDF. The receipt matches a real row of `data/samples/expenses/expenses-2026.csv`, which you will reconcile in Part 6.

---
## Part 1 · One message, many media: standard content blocks

Lesson 01.02 sent an image with the provider's own format (`image_url`). LangChain 1.x also has **standard blocks**, one shape for every provider; the integration converts them:

| Block | Shape | Groq (`ChatGroq`) | Gemini (`ChatGoogleGenerativeAI`) |
|---|---|---|---|
| image | `{"type": "image", "base64": ..., "mime_type": "image/jpeg"}` | ✅ becomes a `data:` URL | ✅ |
| file (PDF) | `{"type": "file", "base64": ..., "mime_type": "application/pdf"}` | ❌ | ✅ native PDF |
| audio | `{"type": "audio", ...}` | ❌ in chat (use Whisper, Lesson 13.02) | ✅ |
| video | `{"type": "video", ...}` | ❌ | ✅ |

Instead of `base64` a block can carry `url` (a public URL) or `file_id` (a provider upload). `mastery.multimodal` builds these blocks from a path, so the MIME type is right on every operating system.

In [ ]:
# 🧪 Offline: build blocks from files and look at them (base64 shortened for printing).
from langchain_core.messages import HumanMessage

from mastery.multimodal import block_bytes, file_block, image_block, text_block

receipt = image_block(SAMPLES / "receipt-photo.jpg")
invoice_pdf = file_block(SAMPLES / "invoice-scanned.pdf")
for block in (receipt, invoice_pdf):
    shown = {k: (v[:24] + "…" if k == "base64" else v) for k, v in block.items()}
    print(shown, f"{block_bytes(block) / 1024:.0f} KB")

message = HumanMessage(content=[text_block("What is the total on this receipt?"), receipt])
print("content blocks:", [b["type"] for b in message.content])

**You should now see** two dicts that start with `'type': 'image'` and `'type': 'file'`, their MIME types and sizes, then `content blocks: ['text', 'image']`.

### What just happened?

A message's `content` is a **list of blocks**: text and media side by side, in the order the model should read them. Base64 makes the file about a third bigger than on disk (`block_bytes` reports the decoded size), and it travels inside every request, so a long chat that keeps an image in its history pays for it on **every turn**.

---
## Part 2 · Check limits before you pay for a 400

Providers reject multimodal requests for reasons a text request never hits. The numbers below were checked on 2026-10-16 and live in `mastery.multimodal.LIMITS` with their source pages:

In [ ]:
# 🧪 Offline: the limits matrix, then three requests checked without sending them.
from mastery.multimodal import LIMITS, LimitError, check_request

table([{"provider": l.provider, "model": l.model, "accepts": ", ".join(sorted(l.accepts)),
        "images/request": l.max_images or "many", "tokens/image": l.image_tokens or "by size",
        "inline MB": l.max_inline_mb, "tokens/PDF page": l.pdf_tokens_per_page or "-",
        "preview": l.preview} for l in LIMITS.values()],
      ["provider", "model", "accepts", "images/request", "tokens/image", "inline MB", "tokens/PDF page", "preview"])
print()
for label, blocks, spec in [
    ("receipt on Groq", [receipt], settings.model_vision),
    ("4 receipts on Groq", [receipt] * 4, "groq"),
    ("PDF on Groq", [invoice_pdf], "groq"),
    ("PDF on Gemini", [invoice_pdf], settings.model_multimodal),
]:
    try:
        est = check_request(blocks, spec)
        print(f"✅ {label:<20} ~{est.media_tokens} media tokens, {est.inline_mb} MB", "; ".join(est.notes))
    except LimitError as error:
        print(f"⛔ {label:<20} refused (expected): {error}")

**You should now see** the two-row matrix, then: receipt on Groq ✅ about 2,048 tokens; 4 receipts ⛔ *refused (expected): … at most 3 per request*; PDF on Groq ⛔ *refused (expected): … does not accept file blocks*; PDF on Gemini ✅ about 258 tokens.

### What just happened?

- On Groq every image costs **2,048 input tokens** whatever its size, and only three fit in one request. A 10-page scan is four requests and about 20K tokens: more than the free tier's 8K tokens per minute for this model, so you would also need to pace the calls.
- Gemini reads the PDF itself at about **258 tokens per page**, scans included. On Gemini 3 models, the text a born-digital PDF already contains is not billed as extra tokens (see the document-processing page).
- The check runs **before** the request: a 400 after uploading 20 MB wastes time, and in a batch job it wastes a quota slot too.

---
## Part 3 · Text layer first: most PDFs don't need a vision model

A PDF exported from software stores the characters themselves. `pypdf` reads them in milliseconds, with no model and no mistakes. A scan stores only a picture. `route_document` decides per page:

In [ ]:
# 🧪 Offline: read the text layer of both invoices and route each page.
from mastery.multimodal import extract_text, route_document

pages = {name: extract_text(SAMPLES / name) for name in ("invoice-digital.pdf", "invoice-scanned.pdf")}
for name, doc in pages.items():
    for route, page in zip(route_document(doc), doc):
        print(f"{name:<20} page {page.page}: {page.chars:4d} chars, {page.images} image(s) → {route.method:<6} ({route.reason})")
print()
print(pages["invoice-digital.pdf"][0].text[:400])

**You should now see** the digital invoice with about 510 characters → `text`, the scan with 0 characters and 1 image → `vision (no text layer (scan))`, then the first lines of the invoice text, from `Nimbus Data Centers S.A.S.` to the line items.

The text is all there but it is not *fields* yet. Turning it into an `InvoiceFields` object is a small **text** task: a fast text model (Lesson 02.03) does it for a few hundred tokens, far cheaper than 2,048 tokens per image. The offline twin below parses it with regular expressions, which works for this one layout only; that brittleness is why you hand it to a model.

In [ ]:
# 🧪 Offline twin: parse the text layer with regular expressions (one vendor layout only).
import re

from mastery.multimodal import InvoiceFields, parse_amount, score_fields

text = pages["invoice-digital.pdf"][0].text
def grab(pattern):
    found = re.search(pattern, text)
    return found.group(1).strip() if found else None

regex_invoice = {
    "vendor": text.splitlines()[0], "invoice_number": grab(r"No\.\s*(\S+)"),
    "date": grab(r"Date:\s*(\S+)"), "due_date": grab(r"Due:\s*(\S+)"), "currency": grab(r"TOTAL\s+([A-Z]{3})"),
    "subtotal": parse_amount(grab(r"Subtotal\s*([\d,.]+)")), "tax": parse_amount(grab(r"VAT 19%\s*([\d,.]+)")),
    "total": parse_amount(grab(r"TOTAL USD\s*([\d,.]+)")),
}
truth = TRUTH["invoice-digital.pdf"]
fields = ["vendor", "invoice_number", "date", "due_date", "currency", "subtotal", "tax", "total"]
scored = score_fields(regex_invoice, truth, fields)
print(regex_invoice)
print("accuracy:", scored["accuracy"], scored["wrong"])
RESULTS.append({"document": "digital PDF", "method": "text layer + regex (🧪)", "media tokens": 0,
                "accuracy": scored["accuracy"], "seconds": "<0.1"})

**You should now see** the eight fields filled in and `accuracy: 1.0 []`.

In [ ]:
# 💸 Text layer → typed fields with the fast text model (a few hundred tokens, no image).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    extractor = get_model("fast", temperature=0).with_structured_output(InvoiceFields, include_raw=True)
    start = time.perf_counter()
    out = extractor.invoke(f"Extract the invoice fields. Dates as YYYY-MM-DD, amounts as plain numbers.\n\n{text}")
    seconds = time.perf_counter() - start
    scored = score_fields(out["parsed"], truth, fields + ["line_items"])
    usage = out["raw"].usage_metadata or {}
    print(out["parsed"].model_dump(exclude={"line_items"}), "| line items:", len(out["parsed"].line_items))
    print(f"accuracy {scored['accuracy']} in {seconds:.1f} s, {usage.get('input_tokens')} input tokens", scored["wrong"])
    RESULTS.append({"document": "digital PDF", "method": f"text layer + {settings.model_fast}", "media tokens": 0,
                    "accuracy": scored["accuracy"], "seconds": round(seconds, 1)})
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 Calls the fast text model once (about 600 input tokens). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

**You should now see** (with a key) the same fields, 4 line items and an accuracy of 1.0 (occasionally 0.89 if the model drops the due date), in about a second.

### What just happened?

For born-digital PDFs the text layer is the source of truth: exact characters, no OCR. The model only has to **structure** text it can already read, which is the cheap, reliable part.

---
## Part 4 · A scan needs vision: Groq `qwen/qwen3.8-27b`

The scanned invoice has no text, so the page goes to a vision model as an image. `page_images` pulls the scan out of the PDF (it *is* the page), and structured output asks for the same `InvoiceFields` schema. `qwen/qwen3.8-27b` is a **preview** model on Groq: fine to learn with, not something to build a production system on without a fallback.

In [ ]:
# 🧪 Offline: get the page image out of the scanned PDF and check the request.
from mastery.multimodal import page_images

(scan_bytes, scan_mime), = page_images(SAMPLES / "invoice-scanned.pdf", page=1)
scan = image_block(scan_bytes, scan_mime)
print(scan_mime, f"{len(scan_bytes) / 1024:.0f} KB,", check_request([scan], settings.model_vision).media_tokens, "image tokens")
PROMPT = ("Extract the invoice fields from this scanned page. Copy the invoice number exactly. "
          "Dates as YYYY-MM-DD, amounts as plain numbers without thousands separators.")

In [ ]:
# 💸 Groq vision: scanned page → InvoiceFields.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    vision = get_model("vision", temperature=0, max_tokens=4096).with_structured_output(InvoiceFields, include_raw=True)
    start = time.perf_counter()
    out = vision.invoke([HumanMessage(content=[text_block(PROMPT), scan])])
    seconds = time.perf_counter() - start
    groq_scan = out["parsed"]
    scored = score_fields(groq_scan, TRUTH["invoice-scanned.pdf"], fields + ["line_items"])
    print(groq_scan.model_dump(exclude={"line_items"}), "| line items:", len(groq_scan.line_items))
    print(f"accuracy {scored['accuracy']} in {seconds:.1f} s, {(out['raw'].usage_metadata or {}).get('input_tokens')} input tokens")
    print("wrong:", scored["wrong"] or "none")
    RESULTS.append({"document": "scanned PDF", "method": f"Groq {settings.model_vision}", "media tokens": 2048,
                    "accuracy": scored["accuracy"], "seconds": round(seconds, 1)})
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 Calls Groq (`qwen/qwen3.8-27b`, preview vision). Each image costs 2,048 input tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

**You should now see** (with a key) the vendor, `INV-2026-0412`, both dates, `USD`, 4 line items and a total of `1197.14`, usually with accuracy 1.0, in a few seconds; input tokens a little over 2,048 (the image plus the prompt and schema).

Without a key, the next cell shows a **simulated** answer so the rest of the lesson has something to score. It is not a model's output: it is the ground truth with three slips that vision models really make on scans (a letter O for a zero, a dropped line, two digits swapped).

In [ ]:
# 🧪 Offline: a SIMULATED vision answer (ground truth + typical scan slips), labelled as such.
simulated = InvoiceFields.model_validate(TRUTH["invoice-scanned.pdf"]).model_copy(deep=True)
simulated.invoice_number = "INV-2O26-0412"          # letter O instead of zero
simulated.line_items = simulated.line_items[:3]      # the last line was missed
simulated.total = 1197.41                            # transposed digits
scored = score_fields(simulated, TRUTH["invoice-scanned.pdf"], fields + ["line_items"])
print("accuracy:", scored["accuracy"])
for problem in scored["wrong"]:
    print(" ✗", problem)
RESULTS.append({"document": "scanned PDF", "method": "simulated vision (🧪)", "media tokens": 2048,
                "accuracy": scored["accuracy"], "seconds": "-"})

**You should now see** `accuracy: 0.667` and three ✗ lines: `invoice_number`, `total` and `line_items`.

### What just happened?

Field-level scoring tells you **which** field breaks, which a single "looks right" never does. In accounts payable a wrong total or invoice number is worse than a missing field: it gets paid. Two cheap guards catch most slips without a second model call: **arithmetic** (line items must add up to the subtotal, subtotal + tax = total) and **formats** (invoice numbers match the vendor's pattern). Exercise 2 adds them.

---
## Part 5 · Gemini reads the PDF itself (free tier)

Gemini takes the PDF as a `file` block: no page images to extract, all pages in one request, about 258 tokens per page. The `multimodal` role reads `MASTERY_MODEL_MULTIMODAL` (default `google_genai:gemini-3.6-flash`, free of charge on the free tier).

In [ ]:
# 💸 Gemini: the scanned PDF itself → InvoiceFields, and the measured token count vs the estimate.
if HAVE_GOOGLE_KEY:
    from mastery.models import get_model

    gemini = get_model("multimodal").with_structured_output(InvoiceFields, include_raw=True)
    for name in ("invoice-scanned.pdf", "invoice-digital.pdf"):
        pdf = file_block(SAMPLES / name)
        start = time.perf_counter()
        out = gemini.invoke([HumanMessage(content=[text_block(PROMPT), pdf])])
        seconds = time.perf_counter() - start
        scored = score_fields(out["parsed"], TRUTH[name], fields + ["line_items"])
        used = (out["raw"].usage_metadata or {}).get("input_tokens")
        print(f"{name}: accuracy {scored['accuracy']} in {seconds:.1f} s; input tokens {used} "
              f"(estimate {check_request([pdf], settings.model_multimodal).media_tokens} for the page + prompt and schema)")
        print("   wrong:", scored["wrong"] or "none")
        RESULTS.append({"document": name.split("-")[1].replace(".pdf", "") + " PDF", "method": f"Gemini {settings.model_multimodal.split(':')[1]} native PDF",
                        "media tokens": used, "accuracy": scored["accuracy"], "seconds": round(seconds, 1)})
else:
    print("⏭️ Skipped: GOOGLE_API_KEY is not set (see the setup cell).")

> 💸 Calls Gemini on the **free tier** (no charge, daily request caps shown in AI Studio). Free-tier content may be used by Google to improve its products: the course sends synthetic samples only. Without `GOOGLE_API_KEY` it prints `⏭️ Skipped`.

**You should now see** (with a key) both PDFs at accuracy 1.0 (or one slip on the scan), each in a few seconds, with input tokens in the hundreds: the page plus the prompt and the schema, far below Groq's 2,048 per image.

### What just happened?

- One request handled the whole file: for a 30-page contract this is the difference between one call and ten.
- **Read `usage_metadata`, don't trust the estimate.** Estimates are for budgets and limits (Part 2); the bill uses the measured count.
- On the free tier, Google may use what you send to improve its products. That is fine for these synthetic invoices and **not** fine for real customer documents: use a paid tier (or another provider) for those.

---
## Part 6 · Real workflow: photographed receipt → expense ledger

An employee photographs a receipt; finance wants to know whether it matches a card charge. The receipt is in Colombian pesos, where `47.600` means forty-seven thousand six hundred: a dot as the **thousands** separator, the opposite of US style. Reading it as 47.6 is the most common, and most expensive, error in this workflow.

In [ ]:
# 🧪 Offline: the ledger, and how reconciliation catches a misread total (SIMULATED slip).
import csv

from mastery.multimodal import ReceiptFields, reconcile_receipt

with (REPO_ROOT / "data" / "samples" / "expenses" / "expenses-2026.csv").open(encoding="utf-8", newline="") as f:
    ledger = list(csv.DictReader(f))
print(len(ledger), "ledger rows; columns:", list(ledger[0]))

correct = ReceiptFields.model_validate(TRUTH["receipt-photo.jpg"])
misread = correct.model_copy(update={"total": 47.6})       # "$ 47.600" read with US conventions
for label, receipt_fields in (("correct read", correct), ("simulated misread", misread)):
    matches = reconcile_receipt(receipt_fields, ledger)
    verdict = "✅ matched" if len(matches) == 1 else ("⚠️ no match → human review" if not matches else "⚠️ possible duplicate")
    print(f"{label:<18} total {receipt_fields.total:>9,.1f} {receipt_fields.currency}: {verdict}", matches[:1])

**You should now see** 128 ledger rows, then the correct read `✅ matched` with the `Carulla Express` row of 2026-07-03 for 47600, and the misread `⚠️ no match → human review`.

In [ ]:
# 💸 Groq vision: the receipt photo → ReceiptFields → reconcile.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    reader = get_model("vision", temperature=0, max_tokens=2048).with_structured_output(ReceiptFields, include_raw=True)
    start = time.perf_counter()
    out = reader.invoke([HumanMessage(content=[
        text_block("Extract the receipt fields. This is a Colombian receipt: '.' separates thousands, so '$ 47.600' is 47600 COP. "
                   "Map 'TARJETA CREDITO' to credit_card."),
        receipt,
    ])])
    seconds = time.perf_counter() - start
    live_receipt = out["parsed"]
    scored = score_fields(live_receipt, TRUTH["receipt-photo.jpg"], ["merchant", "date", "currency", "total", "payment_method"])
    matches = reconcile_receipt(live_receipt, ledger)
    print(live_receipt.model_dump(), f"in {seconds:.1f} s")
    print("accuracy:", scored["accuracy"], "| ledger:", "✅ matched" if len(matches) == 1 else "⚠️ human review")
    RESULTS.append({"document": "receipt photo", "method": f"Groq {settings.model_vision}", "media tokens": 2048,
                    "accuracy": scored["accuracy"], "seconds": round(seconds, 1)})
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 Calls Groq (`qwen/qwen3.8-27b`, preview vision). Each image costs 2,048 input tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

**You should now see** (with a key) `Carulla Express`, `2026-07-03`, `COP`, `47600.0`, `credit_card`, accuracy 1.0 and `✅ matched`. Try removing the sentence about the thousands separator from the prompt and run it again: the total is often still right on this clean photo, but the instruction is what keeps it right on harder ones.

### What just happened?

The vision model is one step of a workflow, and the **ledger is the check**: an exact match closes the receipt automatically, anything else goes to a person. You don't need a perfect extractor; you need one whose mistakes are caught before money moves.

---
## Part 7 · Compare the methods

Every extraction above added a row to `RESULTS`. Rows marked 🧪 are offline (regex or simulated), the others are measured on your key.

In [ ]:
# 🧪 Offline: the comparison table (rows from the cells you ran).
table(RESULTS, ["document", "method", "media tokens", "accuracy", "seconds"])
(OUT / "13-01-extraction-results.json").write_text(json.dumps(RESULTS, indent=2), encoding="utf-8")
print("\nsaved", (OUT / "13-01-extraction-results.json").relative_to(REPO_ROOT))

**You should now see** at least the two 🧪 rows (regex 1.0, simulated vision 0.667); with keys, also the measured rows for Groq and Gemini.

### What just happened?

The decision rule this lesson leaves you with, which `route_document` encodes:

| Page has… | Use | Why |
|---|---|---|
| A text layer | `pypdf` + a fast **text** model | Exact characters, cheapest tokens |
| No text (scan, photo) | A vision model, page by page | Only option; score it and add arithmetic checks |
| Many pages, mixed, or charts | Gemini native PDF | One request per file, ~258 tokens per page |
| Sensitive content | A paid tier with a data agreement | Free tiers may train on your data |

---
## Part 8 · Audio and video are blocks too (preview of 13.02)

The same block shape carries audio and video to Gemini. Before sending, the arithmetic matters more than for images: audio costs **32 tokens per second** on Gemini, so a one-hour call recording is about 115K input tokens in one request. Lesson 13.02 transcribes audio with Groq Whisper instead (billed per hour of audio, 10 s minimum, no tokens) and compares the two.

In [ ]:
# 🧪 Offline: what audio would cost in tokens on Gemini (no audio is sent).
from mastery.multimodal import audio_block, limits_for

gemini_limits = limits_for(settings.model_multimodal)
silence = audio_block(b"RIFF" + b"\x00" * 40, mime_type="audio/wav")   # a placeholder, not real audio
for minutes in (1, 10, 60):
    est = check_request([silence], settings.model_multimodal, audio_seconds=minutes * 60)
    print(f"{minutes:>3} min of audio ≈ {est.media_tokens:>7,} input tokens on Gemini "
          f"({gemini_limits.audio_tokens_per_second} tokens/s)")

**You should now see** about 1,920, 19,200 and 115,200 tokens for 1, 10 and 60 minutes.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `400` from Groq with a PDF | Groq chat models don't accept `file` blocks | Route: text layer, or page images, or Gemini (`check_request` catches it) |
| `400` "too many images" | More than 3 images in one Groq request | Batch pages in threes, pace for the 8K tokens/minute free limit |
| Total off by ×1000 (47.6 vs 47600) | Thousands separator read with the wrong convention | Say the locale in the prompt; reconcile against the ledger |
| `extract_text` returns `''` for a PDF | It is a scan (no text layer) | `route_document` sends it to vision |
| Garbled text from a digital PDF | Fonts without a Unicode map, or a multi-column layout | Compare with the page image; send that page to vision |
| Costs grow every chat turn | The image stays in the message history | Summarise it as text after the first turn, drop the block |
| Different answers on each run | Sampling, or a preview model update | `temperature=0` on Groq; pin the model; keep a scored test set |

## ⚠️ Production notes

> ⚠️ **Production note: text first, vision second, human last.** (Parts 3–6)
> Read text layers for free, send only text-less pages to vision, and validate every extraction with arithmetic and a system of record (ledger, purchase order). Route anything that fails to a person, and log which rule sent it there.

> ⚠️ **Production note: free tiers and documents.** (Part 5)
> Gemini's free tier may use your prompts and files to improve Google's products. Real invoices contain bank accounts and tax IDs: use a paid tier, or an approved provider, before you send them. `qwen/qwen3.8-27b` on Groq is a preview model: keep a fallback and re-run your scored set when it changes.

## 🏋️ Exercises

1. **(Easy)** Add `receipt-photo.jpg` to the Part 2 limits check on Gemini. How many media tokens does `check_request` report, and why does it add a note instead of a number?
2. **(Medium)** Write `invoice_checks(inv)` that returns a list of problems: line items don't add up to the subtotal, or subtotal + tax ≠ total (tolerance 0.01). Run it on the simulated answer from Part 4.
3. **(Medium)** Make `route_document` stricter: send a page to vision if fewer than 60% of its text characters are letters or digits (garbled text layer). Test it on a fake `PageText`.
4. **(Stretch)** Module 06 indexed text documents. Write `vision_loader(pdf_path)` that returns one LangChain `Document` per page: the text layer for `text` pages, and for `vision` pages the text from a vision model (or the stand-in `"[scan: needs OCR]"` offline), with `metadata={"page": n, "method": ...}`.

<details>
<summary><b>Show solutions</b></summary>

**1.** `check_request([receipt], settings.model_multimodal)` reports `0` media tokens and the note *image tokens depend on resolution: read usage_metadata*: Gemini bills images by size and media resolution, so the course doesn't pretend to know the number. Measure it.

**2.**
```python
def invoice_checks(inv, tol=0.01):
    problems = []
    items = round(sum(i.amount for i in inv.line_items), 2)
    if inv.subtotal is not None and abs(items - inv.subtotal) > tol:
        problems.append(f"line items add up to {items}, subtotal says {inv.subtotal}")
    if None not in (inv.subtotal, inv.tax) and abs(inv.subtotal + inv.tax - inv.total) > tol:
        problems.append(f"subtotal + tax = {inv.subtotal + inv.tax:.2f}, total says {inv.total}")
    return problems

print(invoice_checks(simulated))   # both checks fire: 866.0 vs 1006.0, and 1197.14 vs 1197.41
```

**3.**
```python
from mastery.multimodal import PageText

def strict_route(pages, min_chars=200, min_clean=0.6):
    routes = route_document(pages, min_chars=min_chars)
    for i, p in enumerate(pages):
        letters = sum(c.isalnum() for c in p.text)
        if routes[i].method == "text" and letters / max(p.chars, 1) < min_clean:
            routes[i] = type(routes[i])(p.page, "vision", "garbled text layer")
    return routes

print(strict_route([PageText(1, "�$#@" * 100, 400, 0)]))   # → vision, garbled text layer
```

**4.**
```python
from langchain_core.documents import Document

def vision_loader(pdf_path, read_scan=lambda image: "[scan: needs OCR]"):
    doc = extract_text(pdf_path)
    out = []
    for route, page in zip(route_document(doc), doc):
        if route.method == "text":
            content = page.text
        else:
            (data, mime), = page_images(pdf_path, page.page)[:1] or [(b"", "")]
            content = read_scan(image_block(data, mime)) if data else ""
        out.append(Document(page_content=content, metadata={"source": str(pdf_path), "page": page.page, "method": route.method}))
    return out

for d in vision_loader(SAMPLES / "invoice-scanned.pdf"):
    print(d.metadata, d.page_content[:40])
```
With a key, pass `read_scan=lambda block: get_model("vision").invoke([HumanMessage(content=[text_block("Transcribe this page."), block])]).text`.
</details>

## 🔎 Docs drill (5 min)

On Gemini's document-processing page, how many tokens does each PDF page cost, and what happens to the text a born-digital PDF already contains on Gemini 3 models?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Gemini document understanding](https://ai.google.dev/gemini-api/docs/document-processing) and find the section on technical details / token counting.
2. Each page counts as **258 tokens** (it is processed as an image).
3. The same section says the native text extracted from the PDF is **not charged** on Gemini 3 models; check the current wording, since this is the kind of detail that changes.

</details>

## ✅ Checkpoint

- [ ] I built `image` and `file` blocks from paths and know when to use `url` or `file_id` instead.
- [ ] I checked requests against Groq and Gemini limits before sending them.
- [ ] I routed a digital PDF to its text layer and a scan to vision.
- [ ] I extracted typed invoice fields and scored them field by field.
- [ ] I reconciled a receipt with the ledger and sent a mismatch to human review.
- [ ] I can say when native PDF input beats page images, and why free tiers don't get real documents.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open [Groq vision](https://console.groq.com/docs/vision) and [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing). Log in [`docs/research-log.md`](../../docs/research-log.md): which vision models Groq lists today (is `qwen/qwen3.8-27b` still preview?), its image limits, and which Gemini models still show "Free of charge" for input. In AI Studio, note the free-tier requests per day for `gemini-3.6-flash` on your project: that number is your daily document budget.

## Summary

- Standard blocks (`image`, `file`, `audio`, `video`) are one message shape for every provider; check what each provider accepts.
- Check limits and token cost before sending: Groq vision is 3 images and 2,048 tokens per image; Gemini reads PDFs at ~258 tokens per page.
- Text layer first: born-digital PDFs need no vision model.
- Score extractions per field; add arithmetic checks and reconcile against a system of record.
- Free tiers are for synthetic data; real documents need a tier with a data agreement.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 13-02-speech-to-text.ipynb** → [`13-02-speech-to-text.ipynb`](13-02-speech-to-text.ipynb).

[Module 13 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)